# Actividad 3: solución de una EDO de segundo orden con Runge-Kutta

**Módulo III — Simulación de eventos continuos**

## Enunciado

Analice el código presentado en Python para resolver una ecuación diferencial ordinaria mediante el algoritmo de Runge-Kutta. Introduzca una ecuación diferencial ordinaria de segundo orden y resuélvala con el algoritmo proporcionado. Finalmente, formule conclusiones sobre el método numérico y su relación con la simulación de eventos continuos.

## Resultados de aprendizaje

Al finalizar, el estudiante podrá:

- transformar una EDO de segundo orden en un sistema de primer orden;
- implementar el método Runge-Kutta de cuarto orden (RK4);
- interpretar una trayectoria continua a partir de una solución numérica;
- analizar el efecto del tamaño de paso sobre el error y el costo computacional;
- sustentar conclusiones mediante tablas y gráficas.

## 1. Problema aplicado: sistema masa–resorte–amortiguador

Se estudiará el movimiento de una masa conectada a un resorte y a un amortiguador. La ecuación diferencial es:

$$m\frac{d^2x}{dt^2}+c\frac{dx}{dt}+kx=0,$$

donde $x(t)$ es la posición, $m$ la masa, $c$ el coeficiente de amortiguamiento y $k$ la constante del resorte. Se utilizarán las condiciones iniciales:

$$x(0)=1, \qquad x'(0)=0.$$

Para aplicar RK4 se definen dos variables de estado:

$$y_1=x, \qquad y_2=v=\frac{dx}{dt}.$$

Así, la EDO de segundo orden se convierte en el sistema:

$$\frac{dy_1}{dt}=y_2, \qquad \frac{dy_2}{dt}=-\frac{c}{m}y_2-\frac{k}{m}y_1.$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

m = 1.0       # masa (kg)
c = 0.4       # amortiguamiento (N·s/m)
k = 4.0       # constante del resorte (N/m)
x0 = 1.0      # posición inicial (m)
v0 = 0.0      # velocidad inicial (m/s)
t_inicial = 0.0
t_final = 10.0
paso = 0.05

## 2. Algoritmo Runge-Kutta de cuarto orden

Para un sistema $\mathbf{y}'=f(t,\mathbf{y})$, RK4 calcula cuatro pendientes en cada paso:

$$k_1=f(t_n,y_n),$$
$$k_2=f(t_n+h/2,y_n+hk_1/2),$$
$$k_3=f(t_n+h/2,y_n+hk_2/2),$$
$$k_4=f(t_n+h,y_n+hk_3).$$

La siguiente aproximación es:

$$y_{n+1}=y_n+\frac{h}{6}(k_1+2k_2+2k_3+k_4).$$

In [ ]:
def sistema_masa_resorte(t, estado, masa=m, amortiguamiento=c, resorte=k):
    """Devuelve [dx/dt, dv/dt] para el sistema mecánico."""
    posicion, velocidad = estado
    aceleracion = -(amortiguamiento / masa) * velocidad - (resorte / masa) * posicion
    return np.array([velocidad, aceleracion], dtype=float)


def paso_rk4(funcion, t, estado, h):
    """Realiza un paso del algoritmo RK4 para un sistema de EDO."""
    k1 = funcion(t, estado)
    k2 = funcion(t + h/2, estado + h*k1/2)
    k3 = funcion(t + h/2, estado + h*k2/2)
    k4 = funcion(t + h, estado + h*k3)
    return estado + (h/6) * (k1 + 2*k2 + 2*k3 + k4)


def resolver_rk4(funcion, intervalo, estado_inicial, h):
    """Integra un sistema desde t0 hasta tf usando pasos de tamaño h."""
    t0, tf = intervalo
    tiempos = np.arange(t0, tf + h/2, h)
    estados = np.zeros((len(tiempos), len(estado_inicial)), dtype=float)
    estados[0] = estado_inicial
    for i in range(len(tiempos) - 1):
        estados[i + 1] = paso_rk4(funcion, tiempos[i], estados[i], h)
    return tiempos, estados

### Análisis del código

- `sistema_masa_resorte` representa el modelo matemático y calcula la velocidad de cambio de cada variable de estado.
- `paso_rk4` combina cuatro estimaciones de la pendiente para avanzar desde $t_n$ hasta $t_{n+1}$.
- `resolver_rk4` repite el paso numérico y almacena toda la trayectoria temporal.
- El algoritmo es general: puede resolver otros sistemas si se reemplaza la función del modelo y el vector inicial.

In [ ]:
tiempo, solucion = resolver_rk4(
    sistema_masa_resorte,
    (t_inicial, t_final),
    np.array([x0, v0]),
    paso,
)

posicion = solucion[:, 0]
velocidad = solucion[:, 1]
resultados = pd.DataFrame({
    'tiempo': tiempo,
    'posición': posicion,
    'velocidad': velocidad,
})
resultados.head(10)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))

ejes[0].plot(tiempo, posicion, label='Posición x(t)', color='#1565C0')
ejes[0].plot(tiempo, velocidad, label='Velocidad v(t)', color='#D84315', alpha=0.85)
ejes[0].axhline(0, color='black', linewidth=0.7)
ejes[0].set(xlabel='Tiempo (s)', ylabel='Respuesta', title='Evolución temporal del sistema')
ejes[0].grid(alpha=0.3)
ejes[0].legend()

ejes[1].plot(posicion, velocidad, color='#6A1B9A')
ejes[1].scatter([x0], [v0], color='black', label='Estado inicial', zorder=3)
ejes[1].set(xlabel='Posición (m)', ylabel='Velocidad (m/s)', title='Diagrama de fase')
ejes[1].grid(alpha=0.3)
ejes[1].legend()

plt.tight_layout()
plt.show()

## 3. Validación con la solución analítica

Para los parámetros seleccionados, el sistema es subamortiguado. Su solución exacta permite comprobar la precisión del algoritmo. Esta comparación es una estrategia de validación; en problemas reales complejos normalmente no se dispone de una solución analítica.

In [ ]:
omega_n = np.sqrt(k / m)
zeta = c / (2 * np.sqrt(k * m))
omega_d = omega_n * np.sqrt(1 - zeta**2)

posicion_exacta = np.exp(-zeta * omega_n * tiempo) * (
    x0 * np.cos(omega_d * tiempo)
    + ((v0 + zeta * omega_n * x0) / omega_d) * np.sin(omega_d * tiempo)
)
error = np.abs(posicion - posicion_exacta)

print(f'Factor de amortiguamiento ζ: {zeta:.4f}')
print(f'Error máximo de posición: {error.max():.3e}')

plt.figure(figsize=(9, 4.5))
plt.plot(tiempo, posicion_exacta, label='Solución analítica', linewidth=2.5)
plt.plot(tiempo, posicion, '--', label='Solución RK4', linewidth=1.5)
plt.xlabel('Tiempo (s)')
plt.ylabel('Posición (m)')
plt.title('Validación de la solución numérica')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Efecto del tamaño de paso

Un paso pequeño suele mejorar la precisión, pero incrementa el número de cálculos. Ejecute la siguiente comparación y analice el compromiso entre error y costo computacional.

In [ ]:
comparacion = []
for h in [0.5, 0.25, 0.1, 0.05, 0.025]:
    t_h, y_h = resolver_rk4(
        sistema_masa_resorte, (t_inicial, t_final), np.array([x0, v0]), h
    )
    exacta_h = np.exp(-zeta * omega_n * t_h) * (
        x0 * np.cos(omega_d * t_h)
        + ((v0 + zeta * omega_n * x0) / omega_d) * np.sin(omega_d * t_h)
    )
    comparacion.append({
        'paso h': h,
        'número de pasos': len(t_h) - 1,
        'error máximo': np.max(np.abs(y_h[:, 0] - exacta_h)),
    })

tabla_error = pd.DataFrame(comparacion)
tabla_error

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.loglog(tabla_error['paso h'], tabla_error['error máximo'], marker='o')
plt.xlabel('Tamaño de paso h')
plt.ylabel('Error máximo')
plt.title('Convergencia de RK4')
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Desarrollo que debe entregar el estudiante

1. Explique con sus propias palabras qué representan `k1`, `k2`, `k3` y `k4`.
2. Justifique la transformación de la EDO de segundo orden en dos ecuaciones de primer orden.
3. Ejecute el caso base y describa el comportamiento de la posición y la velocidad.
4. Cambie el amortiguamiento a $c=0$, $c=1$ y $c=4$. Genere y compare las gráficas.
5. Modifique el tamaño de paso y explique su efecto sobre el error y el número de operaciones.
6. Proponga otra EDO de segundo orden de un contexto real, defina parámetros y condiciones iniciales, adáptela al algoritmo y presente sus resultados.
7. Entregue el notebook ejecutado con tablas, gráficas, interpretación y conclusiones.

### Ejemplos de ecuaciones alternativas

- Péndulo no lineal: $\theta''+(g/L)\sin(\theta)=0$.
- Caída con resistencia: $y''=g-(b/m)y'$.
- Circuito RLC: $LCq''+RCq'+q=V(t)$.

> **Importante:** no basta con cambiar la ecuación. Deben explicarse las variables, unidades, parámetros, condiciones iniciales y significado del resultado.

## 6. Orientación para las conclusiones

Redacte entre tres y cinco conclusiones sustentadas en los resultados. Considere:

- precisión observada y criterio utilizado para validarla;
- relación entre tamaño de paso, error y costo computacional;
- utilidad de transformar una EDO de orden superior en un sistema de primer orden;
- interpretación física de la trayectoria y del diagrama de fase;
- relación con la simulación continua: el estado cambia continuamente en el tiempo, aunque el computador lo aproxima en instantes discretos;
- limitaciones del método, como pasos inadecuados, acumulación de error o sistemas rígidos.

### Conclusión de referencia

El método RK4 permitió aproximar con alta precisión la respuesta continua del sistema masa–resorte–amortiguador. Al reducir el tamaño de paso disminuyó el error, aunque aumentó el número de evaluaciones del modelo. La simulación mostró cómo la energía se disipa debido al amortiguamiento hasta que el sistema se aproxima al equilibrio. Aunque el fenómeno modelado evoluciona continuamente, su trayectoria se obtiene computacionalmente mediante una sucesión de aproximaciones temporales.

## 7. Criterios de evaluación

| Criterio | Porcentaje |
|---|---:|
| Análisis del algoritmo RK4 | 20 % |
| Formulación de la EDO de segundo orden | 20 % |
| Implementación y ejecución correcta | 25 % |
| Gráficas, validación y análisis del error | 20 % |
| Conclusiones sobre el método y la simulación continua | 15 % |